# Phase 3: Modular initialization, Fishers, and a small MCMC chain

This is an **architectural smoke tutorial**, not a production forecast. Its tiny
Fisher and nested-sampling settings demonstrate the workflow only; they do not
establish convergence or scientific constraints.


In [ ]:
from __future__ import annotations
import os
import shutil
from pathlib import Path
import numpy as np
from cosmicfishpie.configs.context import build_analysis_context
from cosmicfishpie.cosmology.cosmology import cosmo_functions
from cosmicfishpie.fishermatrix.cosmicfish import FisherMatrix
from cosmicfishpie.likelihood import NautilusSampler, PhotometricLikelihood
from cosmicfishpie.likelihood.sampler import load_chain_metadata

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / "notebooks/results/phase3_modular_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Tutorial Fisher output: {OUTPUT_DIR.resolve()}")


## The Phase 3 ownership model

```text
user options + fiducials
        -> build_analysis_context(...)
        -> AnalysisContext (immutable run-local snapshot)
        -> cosmology / likelihood / sampler with configuration=context
```

`AnalysisContext` owns resolved settings, observables, fiducials, backend
selection, and backend parameters. Runtime objects bind to that context at
construction. `FisherMatrix` remains for forecasts, not as mandatory setup for
likelihoods or samplers.


In [ ]:
FIDUCIAL = {"Omegam": 0.32, "Omegab": 0.05, "h": 0.67, "ns": 0.96, "sigma8": 0.815583}
FREEPARS = {"Omegam": 0.01, "sigma8": 0.01}
OPTIONS = {
    "accuracy": 1, "ell_sampling": 12, "feedback": 0, "code": "symbolic",
    "nonlinear": True, "survey_name": "Euclid",
    "survey_name_photo": "Euclid-Photometric-ISTF-Pessimistic",
    "survey_name_spectro": False, "cosmo_model": "LCDM",
}
context = build_analysis_context(
    fiducialpars=FIDUCIAL, freepars=FREEPARS, options=OPTIONS,
    observables=["WL"], survey_name="Euclid", cosmo_model="LCDM",
)
print("backend:", context.input_type)
print("observables:", context.observables)
print("nonlinear:", context.settings["nonlinear"])
print("fiducials:", dict(context.fiducialcosmopars))


## Direct likelihood initialization

This uses the explicit context directly; no `FisherMatrix` is created merely to
carry setup state.


In [ ]:
likelihood = PhotometricLikelihood(cosmo_data=context, cosmo_theory=context)
theory = likelihood.compute_theory({})
print("theory cells:", sorted(theory))
assert theory


## A small Fisher calculation

The Fisher API remains the correct tool for forecasts. This compact example
writes only to the ignored notebook-results directory.


In [ ]:
fisher_engine = FisherMatrix(
    fiducialpars=FIDUCIAL, freepars=FREEPARS,
    options={**OPTIONS, "results_dir": f"{OUTPUT_DIR}/", "outroot": "phase3_demo_"},
    observables=["WL"], surveyName="Euclid", cosmoModel="LCDM",
)
fisher = fisher_engine.compute()
print("Fisher output:", fisher.file_name)
print("Fisher parameters:", fisher.param_names)
print("Fisher shape:", fisher.fisher_matrix.shape)


## A deliberately short MCMC smoke chain

`NautilusSampler` creates an `AnalysisContext` internally, then constructs the
likelihood from it. The `n_eff` and `n_like_max` limits make this a real but
short smoke chain. It is not a converged posterior.


In [ ]:
CHAIN_NAME = "phase3_modular_demo"
CHAIN_DIR = REPO_ROOT / "chains" / f"chains_{CHAIN_NAME}"
shutil.rmtree(CHAIN_DIR, ignore_errors=True)
sampler = NautilusSampler({
    "name": CHAIN_NAME, "fiducial": FIDUCIAL, "observables": ["WL"],
    "options": dict(OPTIONS),
    "priors": {"Omegam": [0.30, 0.34], "sigma8": [0.79, 0.84]},
    "sampler_settings": {
        "n_live": 10, "n_networks": 1, "n_batch": 4, "pool": 1,
        "n_eff": 12, "n_like_max": 48, "verbose": False,
    },
})
sampler.run()
chain_file, sampled_fiducial, metadata, labels = load_chain_metadata(str(CHAIN_DIR))
chain = np.atleast_2d(np.loadtxt(chain_file))
print("chain:", chain_file)
print("sampled parameters:", list(sampled_fiducial))
print("samples:", len(chain))
n_parameters = len(labels)
weights = chain[:, n_parameters]
posterior_means = np.average(chain[:, :n_parameters], axis=0, weights=weights)
print("weighted posterior means:", dict(zip(labels, posterior_means)))


## A/B isolation demonstration

The authoritative regression is `tests/runtime_context_test.py`. Here, a
runtime bound to A stays bound to A after construction of a conflicting B.


In [ ]:
runtime_a = cosmo_functions(FIDUCIAL, configuration=context)
hubble_a = float(runtime_a.Hubble(0.0))
context_b = build_analysis_context(
    fiducialpars={**FIDUCIAL, "Omegam": 0.29}, freepars=FREEPARS,
    options={**OPTIONS, "nonlinear": False}, observables=["WL"],
    survey_name="Euclid", cosmo_model="LCDM",
)
assert runtime_a.configuration is context
assert runtime_a.settings["nonlinear"] is True
assert runtime_a.fiducialcosmopars["Omegam"] == 0.32
assert np.isfinite(hubble_a)
print("A:", runtime_a.settings["nonlinear"], runtime_a.fiducialcosmopars["Omegam"])
print("B:", context_b.settings["nonlinear"], context_b.fiducialcosmopars["Omegam"])


## Next steps

- Use `AnalysisContext` directly for likelihoods, samplers, and custom apps.
- Use `FisherMatrix` for Fisher forecasts and legacy-compatible workflows.
- Derivatives run through a backend-neutral `DerivativeProvider`; `FisherMatrix(..., derivative_provider=provider)` can inject a future autodiff implementation without coupling probe covariance code to JAX.
- Phase 3 still ships only the finite-difference provider. A JAX provider requires a genuinely traceable backend/emulator plus explicit handling for mixed differentiable and nuisance parameters.
- The legacy resolver still initializes global state while *building* a context;
  Phase 3 protects runtime objects constructed from the resulting snapshots.

Fisher artifacts: `notebooks/results/phase3_modular_demo/`. Sampler artifacts:
`chains/chains_phase3_modular_demo/`. Both paths are ignored by Git.

```bash
uv run jupyter nbconvert --to notebook --execute \
  notebooks/phase3_modular_initialization_fisher_mcmc.ipynb \
  --output phase3_modular_initialization_fisher_mcmc.executed.ipynb \
  --ExecutePreprocessor.timeout=900
uv run pytest tests/runtime_context_test.py tests/cmb_context_test.py -q
```
